In [1]:
from pathlib import Path
import sys

import matplotlib
import numpy as np
import pandas as pd
import scipy
import statsmodels

print("Python:", sys.executable)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("statsmodels:", statsmodels.__version__)
print("matplotlib:", matplotlib.__version__)

Python: C:\Users\Administrator\Documents\Development\ai-software-quality-pipeline\.venv-analysis\Scripts\python.exe
pandas: 3.0.5
numpy: 2.5.1
scipy: 1.18.0
statsmodels: 0.14.6
matplotlib: 3.11.1


In [3]:
from pathlib import Path
import pandas as pd

analysis_dir = Path.cwd().parent

runs_path = analysis_dir / "outputs" / "processed_runs.csv"
iterations_path = analysis_dir / "outputs" / "processed_iterations.csv"
integrity_path = analysis_dir / "tables" / "integrity_checks.csv"
dictionary_path = analysis_dir / "tables" / "data_dictionary.csv"

runs = pd.read_csv(
    runs_path,
    parse_dates=["startedAt", "completedAt"],
)

iterations = pd.read_csv(iterations_path)
integrity = pd.read_csv(integrity_path)
data_dictionary = pd.read_csv(dictionary_path)

print("Runs:", runs.shape)
print("Iterations:", iterations.shape)
print("Integrity checks:", integrity.shape)
print("Data dictionary:", data_dictionary.shape)

Runs: (90, 86)
Iterations: (125, 13)
Integrity checks: (38, 3)
Data dictionary: (86, 8)


In [6]:
integrity

assert integrity["passed"].all()
assert len(runs) == 90
assert runs["experimentRunId"].is_unique
assert runs["runId"].is_unique

print("Notebook integrity review: PASSED")

Notebook integrity review: PASSED


In [7]:
design_counts = pd.crosstab(
    runs["specificationId"],
    runs["workflow"],
)

design_counts

workflow,automated-refinement,one-shot,quality-focused
specificationId,,,
booking,10,10,10
quiz,10,10,10
todo,10,10,10


In [12]:
classification_counts = (
    runs.groupby(
        ["workflow", "resultClassification"],
        observed=True,
    )
    .size()
    .unstack(fill_value=0)
)

classification_counts

resultClassification,valid-passed,valid-poor-quality
workflow,,
automated-refinement,30,0
one-shot,2,28
quality-focused,19,11


In [13]:
missing_summary = (
    runs.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

missing_summary[missing_summary["missing_count"] > 0]

,missing_count
failureStage,90
failureType,90
error,90
converged,60
automatedRefinementUsed,60


The null values in `converged` are structural rather than missing outcome data.
They occur for one-shot and quality-focused runs, where convergence is not
applicable. Zero values for issue counts, failed tests, refinement iterations
and runtime errors are retained as legitimate recorded outcomes.

In [14]:
automated = runs.loc[
    runs["workflow"] == "automated-refinement"
].copy()

automated["refinementIterations"].value_counts().sort_index()

refinementIterations
0     1
1    23
2     6
Name: count, dtype: int64

In [15]:
runs[
    [
        "experimentRunId",
        "sequence",
        "specificationId",
        "workflow",
        "repetition",
        "finalAllPassed",
        "workflowDurationMs",
        "aiGenerationDurationMs",
        "totalTokens",
        "refinementIterations",
    ]
].head(10)

,experimentRunId,sequence,specificationId,workflow,repetition,finalAllPassed,workflowDurationMs,aiGenerationDurationMs,totalTokens,refinementIterations
0,booking__quality-focused__10,1,booking,quality-focused,10,True,36800,32367,4037,0
1,booking__quality-focused__4,2,booking,quality-focused,4,False,21660,19224,3869,0
2,booking__automated-refinement__6,3,booking,automated-refinement,6,True,54469,44246,12581,1
3,todo__quality-focused__10,4,todo,quality-focused,10,True,21986,20045,3703,0
4,todo__automated-refinement__7,5,todo,automated-refinement,7,True,32344,27947,3556,0
5,quiz__automated-refinement__8,6,quiz,automated-refinement,8,True,57901,49472,17029,2
6,booking__one-shot__5,7,booking,one-shot,5,False,27169,23884,4113,0
7,quiz__one-shot__3,8,quiz,one-shot,3,False,19369,16457,3379,0
8,booking__quality-focused__9,9,booking,quality-focused,9,True,42198,39839,4029,0
9,todo__automated-refinement__9,10,todo,automated-refinement,9,True,42385,34443,10603,1
